In [1]:
from dataset_cfd import *
from config import *
import torch

In [3]:
cfd_dataset = CFD_Dataset(root = "Data_with_P", patch_size = 8, grid_size = 64)

In [16]:
for i, patches in enumerate(cfd_dataset.patches_list):
    data = patches[:, :-FOURIER_DIMENSIONS]  # (num_patches, C*ph*pw)
    pw = 8 * 8  # 64 values per channel per patch
    u_vals = data[:, :pw]
    v_vals = data[:, pw:2*pw]
    P_vals = data[:, 2*pw:]
    
    if P_vals.abs().max() > 700:
        print(f"Sample {i}, Re={cfd_dataset.re_list[i]:.1f}")
        print(f"  u: [{u_vals.min():.2f}, {u_vals.max():.2f}]")
        print(f"  v: [{v_vals.min():.2f}, {v_vals.max():.2f}]")
        print(f"  P: [{P_vals.min():.2f}, {P_vals.max():.2f}]")

Sample 223, Re=0.1
  u: [-0.93, 4.26]
  v: [-1.98, 1.97]
  P: [-793.64, 794.08]


In [18]:
for i, patches in enumerate(cfd_dataset.patches_list):
    data = patches[:, :-FOURIER_DIMENSIONS]
    pw = 8 * 8
    P_vals = data[:, 2*pw:]
    if P_vals.abs().max() > 10:
        print(f"Re={cfd_dataset.re_list[i]:.4f}, P range: [{P_vals.min():.2f}, {P_vals.max():.2f}]")

Re=0.1000, P range: [-793.64, 794.08]


In [119]:
# split sizes
train_size = int(TRAIN_SPLIT * len(cfd_dataset))
test_size  = len(cfd_dataset) - train_size

# random split
train_dataset, test_dataset = torch.utils.data.random_split(
    cfd_dataset,
    [train_size, test_size]
)

# dataloaders
train_dataloader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False
)


for src, tgt in test_dataloader:
    print("tgt mean:", tgt.mean().item())
    print("tgt std:", tgt.std().item())
    print("tgt min:", tgt.min().item())
    print("tgt max:", tgt.max().item())
    break

print("\n")

for src, tgt in train_dataloader:
    print("tgt mean:", tgt.mean().item())
    print("tgt std:", tgt.std().item())
    print("tgt min:", tgt.min().item())
    print("tgt max:", tgt.max().item())
    break

tgt mean: 0.024361861869692802
tgt std: 0.8274297118186951
tgt min: -3.6086461544036865
tgt max: 4.365202903747559


tgt mean: 0.026417460292577744
tgt std: 0.8009656071662903
tgt min: -21.090473175048828
tgt max: 22.658079147338867


In [121]:
print("P_std_list:", sorted(cfd_dataset.P_std_list)[:10])


P_std_list: [0.050953176, 0.050983764, 0.05101457, 0.051045537, 0.051076766, 0.051108155, 0.05113977, 0.05117157, 0.051203527, 0.05123562]
